## Issue:

- considering shift parameter storage:
	- central is that the user ends up with
		- i) the time series data set -> to feed into detector
		- ii) information about where the true shifts are -> to verify the detectors output
	- most shift parameters might not be relevant in the end
	- parameter storage currently also takes up time
	- -> way simpler version might suffice
	- -> for detector validation the user needs:
		- where did the shift happen?
		- when did the shift happen?
		- (what kind of shift happened?)
	- best solution might be to store the time series and the shift information in the same DataSet (time x lat x lon)
		- shift information is reduced to a 3D boolean mask
		- each shift is stored in a seperate mask
		- advantage: good compatibility with TOAD -> easy validation
	- Alternative: again both information in same DataSet, but store shift id in one DataArray only. Contains lists which lengths represents the number of shifts. If no shift occurs -> 0 entry

## To Do:

- check how TOAD is storing the cluster information (output)
- write both approaches
- check which one is more efficient
- check which one is more convenient for users

## TOAD Cluster Data Storage

![Shift Parameter Storage Diagram](img/td_data.png)
![Shift Parameter Storage Diagram](img/thk_cluster.png)

TOAD stores the determined cluster id's in the xarray.DataSet in a single,distinct DataArray. This array has the dimensions (time x lat x lon). Currently, there is no way in storing more than one cluster id for an single space-time-coordinate. If there is no cluster assigned to a space-time-coordinate, it defaults to "-1".


## Approach 1

PONDS stores the information of **when** and **where** **which** shift was used in a (time x lat x lon) boolean mask. Each distinct shift has its distinct mask. Hence, the final xarray.DataSet contains one DataArray for the time series data and as many boolean masks as shifts where applied.

In [13]:
import numpy as np
import xarray as xr

lat_size = 20
lon_size = 20
time_size = 300

# 1) Set up coordinates
lats = np.linspace(-90, 90, lat_size)
lons = np.linspace(-180, 180, lon_size)
time = np.linspace(0, time_size, time_size)

# 2) Generate background data with trend and noise
shape = (time_size, lat_size, lon_size)
data_ts = np.zeros(shape)

# - Create xarray DataSet
ds = xr.Dataset(
    {
        "ts": (["time", "lat", "lon"], data_ts)
    },
    coords={
        "time": time,
        "lat": lats,
        "lon": lons
    }
)

def append_by_mask(ds, mask, true_value=1, false_value=-1, name="c1"):
    """
    Appends a new DataArray to the xarray Dataset based on a boolean mask.
    """
    # Create a new DataArray with the same shape as the mask
    new_da = xr.DataArray(
        np.full(mask.shape, false_value, dtype=int),
        dims=ds.dims,
        coords=ds.coords
    )
    
    # Set values where mask is True
    new_da.values[mask] = true_value
    
    # Append the new DataArray to the Dataset
    ds[name] = new_da

# Example usage with a random masks
mask1 = np.random.rand(*shape) > 0.5
#append_by_mask(ds, mask1)

mask2 = np.random.rand(*shape) > 0.5
#append_by_mask(ds, mask2, true_value=2, name="c2")  # Example with different values

mask3 = np.random.rand(*shape) > 0.5
#append_by_mask(ds, mask3, true_value=1, name="c3")  # Example with different values

mask4 = np.random.rand(*shape) > 0.5
#append_by_mask(ds, mask4, true_value=4, name="c4")  # Example with different values

print("\nDataArray after appending based on mask:")
print(ds)


DataArray after appending based on mask:
<xarray.Dataset> Size: 963kB
Dimensions:  (time: 300, lat: 20, lon: 20)
Coordinates:
  * time     (time) float64 2kB 0.0 1.003 2.007 3.01 ... 297.0 298.0 299.0 300.0
  * lat      (lat) float64 160B -90.0 -80.53 -71.05 -61.58 ... 71.05 80.53 90.0
  * lon      (lon) float64 160B -180.0 -161.1 -142.1 ... 142.1 161.1 180.0
Data variables:
    ts       (time, lat, lon) float64 960kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0


In [23]:
a = np.array([-1,0,0,-1])
ids = np.unique(a)
ids[ids >= 0]


array([0])

In [14]:
# get all variable names
variable_names = list(ds.data_vars.keys())
print("\nVariable names in the Dataset:")

print(variable_names)

# only keep names that start with 'c'
variable_names = [name for name in variable_names if name.startswith('c')]
print("\nFiltered variable names (starting with 'c'):")
print(variable_names)

# sort variable names
variable_names.sort()
print("\nSorted variable names:")
print(variable_names)

# get the last variable name's index
last_variable_index = variable_names[-1]
print("\nLast variable name index:")
print(last_variable_index)


Variable names in the Dataset:
['ts']

Filtered variable names (starting with 'c'):
[]

Sorted variable names:
[]


IndexError: list index out of range

In [13]:
# Extract the cluster masks from the dataset
def extract_masks(ds):
    """
    Extracts all DataArrays from the Dataset that are not called 'ts'.
    """
    masks = []
    for name, da in ds.data_vars.items():
        if name != "ts":  # Exclude the time series
            da_mask = np.zeros(da.shape, dtype=bool)
            mask_ix = np.argwhere(da.values > 0)
            da_mask[mask_ix[:, 0], mask_ix[:, 1], mask_ix[:, 2]] = True

            masks.append(da_mask)
            #print(f"Extracting mask for {name}: {da_mask}")

    return np.array(masks)

# Extract masks from the dataset
cluster_masks = extract_masks(ds)

# check if the cluster mask are identical to the original masks
for i, mask in enumerate([mask1, mask2, mask3, mask4]):
    print(f"Mask {i+1} matches cluster mask {i+1}: {np.array_equal(cluster_masks[i], mask)}")

Mask 1 matches cluster mask 1: True
Mask 2 matches cluster mask 2: True
Mask 3 matches cluster mask 3: True
Mask 4 matches cluster mask 4: True


## Approach 2

PONDS stores the information of **when** and **where** **which** shift was used in a (time x lat x lon) DataArray. Each coordinated contains a list which length depends on the total number of shift-blobs (a.k.a. cluster) in the PONDS object. If a new blob is added, all lists receive a new entry. If the space-time-coordinate is NOT part of the cluster, the entry is -1. If it is part of the cluster, the entry is the shift id corresponding to the type of shift.

In [14]:
import numpy as np
import xarray as xr

lat_size = 20
lon_size = 20
time_size = 300

# 1) Set up coordinates
lats = np.linspace(-90, 90, lat_size)
lons = np.linspace(-180, 180, lon_size)
time = np.linspace(0, time_size, time_size)

# 2) Generate background data with trend and noise
data_ts = np.zeros((time_size, lat_size, lon_size))

# 3) Set up empty lists for shift id
shape = (time_size, lat_size, lon_size)
# Initialize a numpy array of empty lists
data = np.empty(shape, dtype=object)
# Fill with empty lists
for index, _ in np.ndenumerate(data):
    data[index] = []

# 4) Create xarray DataSet containing both time series and shift info
ds = xr.Dataset(
    {
        "ts": (["time", "lat", "lon"], data_ts),
        "shift_id": (["time", "lat", "lon"], data)
    },
    coords={
        "time": time,
        "lat": lats,
        "lon": lons
    }
)

# Define the function to append values based on a boolean mask
def append_by_mask(ds, mask, true_value=1, false_value=-1):
    """
    Appends true_value or false_value to each list in da depending on mask.
    
    da: xarray.DataArray with dtype=object containing lists
    mask: numpy array or xarray.DataArray with same shape as da, dtype=bool
    true_value: value to append if mask is True
    false_value: value to append if mask is False
    """
    for index, val in np.ndenumerate(mask):
        if val:
            ds["shift_id"].values[index].append(true_value)
        else:
            ds["shift_id"].values[index].append(false_value)

# Example usage with a random masks
mask1 = np.random.rand(*shape) > 0.5
append_by_mask(ds, mask1)

mask2 = np.random.rand(*shape) > 0.5
append_by_mask(ds, mask2, true_value=2)  # Example with different values

mask3 = np.random.rand(*shape) > 0.5
append_by_mask(ds, mask3, true_value=1)  # Example with different values

mask4 = np.random.rand(*shape) > 0.5
append_by_mask(ds, mask4, true_value=4)  # Example with different values

print("\nDataArray after appending based on mask:")
print(ds)


DataArray after appending based on mask:
<xarray.Dataset> Size: 2MB
Dimensions:   (time: 300, lat: 20, lon: 20)
Coordinates:
  * time      (time) float64 2kB 0.0 1.003 2.007 3.01 ... 298.0 299.0 300.0
  * lat       (lat) float64 160B -90.0 -80.53 -71.05 -61.58 ... 71.05 80.53 90.0
  * lon       (lon) float64 160B -180.0 -161.1 -142.1 ... 142.1 161.1 180.0
Data variables:
    ts        (time, lat, lon) float64 960kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    shift_id  (time, lat, lon) object 960kB [-1, 2, -1, -1] ... [1, -1, -1, 4]


In [15]:
# extract cluster mask
def extract_cluster_mask(ds, false_value=-1):
    """
    Extracts a boolean mask for each entry in da.
    """
    masks = []
    for i in range(len(ds["shift_id"].values[0, 0, 0])):
        masks.append(np.zeros(ds["shift_id"].shape, dtype=bool))
        for index, val in np.ndenumerate(ds["shift_id"].values):
            if val[i] != false_value:
                masks[i][index] = True
    return np.array(masks)

# Example usage of extract_cluster_mask
cluster_mask = extract_cluster_mask(ds)
print("\nExtracted cluster mask shape:")

print(cluster_mask.shape)
print("\n")

# check if the cluster mask are identical to the original masks
for i, mask in enumerate([mask1, mask2, mask3, mask4]):
    print(f"Mask {i+1} matches cluster mask {i+1}: {np.array_equal(cluster_mask[i], mask)}")


Extracted cluster mask shape:
(4, 300, 20, 20)


Mask 1 matches cluster mask 1: True
Mask 2 matches cluster mask 2: True
Mask 3 matches cluster mask 3: True
Mask 4 matches cluster mask 4: True
